In [70]:
import arxiv
import tempfile
import time
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [71]:
def verify_results(results, author=None, 
                   title=None, category=None, 
                   exclude_words=None, abstract=None,):
    
    results_list = []  # Store results for later validation

    for result in results:
        # Extract key paper details
        paper_title = result.title
        paper_author_list = [author.name for author in result.authors]
        paper_abstract = result.summary
        paper_category = result.primary_category

        # Print paper details
        print(f"\nTitle: {paper_title}")
        print(f"- Author(s): {', '.join(paper_author_list)}")
        print(f"- Category: {paper_category}")
        print(f"- Abstract (first 200 chars): {paper_abstract[:200]}...")

        # Check if each filter is matched or not
        match_results = {
            "Author Match": "✅" if author is None or any(author.lower() in a.lower() for a in paper_author_list) else "❌",
            "Title Match": "✅" if title is None or title.lower() in paper_title.lower() else "❌",
            "Category Match": "✅" if category is None or category in paper_category else "❌",
            "Abstract Match": "✅" if abstract is None or abstract.lower() in paper_abstract.lower() else "❌",
            "Exclusion (NOT)": "✅" if exclude_words is None or exclude_words.lower() not in paper_title.lower() else "❌"
        }

        # Print filter match results
        print("\nFilter Matches:")
        for filter_name, outcome in match_results.items():
            print(f"- {filter_name}: {outcome}")
        
        print("-" * 100)

        # Store result if it matches the criteria
        results_list.append(result) if "❌" not in match_results.values() else None
    
    print(f"\nFinal Papers Matching All Filters: {len(results_list)}\n")
    return results_list

In [ ]:
def format_query(query="LLM", author=None, title=None, 
                 category=None, exclude_words=None, 
                 abstract=None, journal_ref=None, 
                 doi=None, sort_by="relevance"):

    # Base query
    search_query_parts = [query]  

    if author:
        search_query_parts.append(f'au:"{author}"') 

    if title:
        search_query_parts.append(f'ti:"{title}"') 

    if category:
        search_query_parts.append(f'cat:"{category}"')  

    if abstract:
        search_query_parts.append(f'abs:"{abstract}"') 

    if journal_ref:
        search_query_parts.append(f'jr:"{journal_ref}"') 

    if doi:
        search_query_parts.append(f'doi:"{doi}"') 

    if exclude_words:
        search_query_parts.append(f'NOT "{exclude_words}"') 

    # Combine filters with AND
    search_query = " AND ".join(search_query_parts)

    # Print applied filters
    print("\nFilters:")
    print(f"- Query: {query}")
    print(f"- Author: {author or 'None'}")
    print(f"- Title must contain: {title or 'None'}")
    print(f"- Category: {category or 'None'}")
    print(f"- Exclude words: {exclude_words or 'None'}")
    print(f"- Abstract must contain: {abstract or 'None'}")
    print(f"- Journal Reference: {journal_ref or 'None'}")
    print(f"- DOI: {doi or 'None'}")
    print(f"- Sort By: {sort_by}")
    print(f"- Full Query: {search_query}\n")
    print("-" * 100)

    return search_query

In [ ]:
def fetch_papers(query="LLM", max_results=10,
                  author=None, title=None, 
                  category=None, exclude_words=None, 
                  abstract=None, journal_ref=None, 
                  doi=None, sort_by="relevance"):
    
    search_query = format_query(query=query, 
                                author=author, title=title, 
                                category=category, exclude_words=exclude_words, 
                                abstract=abstract, journal_ref=journal_ref, 
                                doi=doi, sort_by="relevance")

    # Create ArXiv search object
    client = arxiv.Client()
    search = arxiv.Search(
        query=search_query,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.Relevance if sort_by == "relevance" else arxiv.SortCriterion.SubmittedDate
    )

    results = client.results(search)

    results_list = verify_results(results, 
                                author=author, title=title, 
                                category=category, exclude_words=exclude_words, 
                                abstract=abstract)
    
    return results_list

In [ ]:
results = fetch_papers(query="Artificial Intelligence", 
             max_results=3, 
             exclude_words="Reinforcement Learning",
             author="Yann LeCun", 
             abstract="Machine Learning",
             category="cs.LG")

print(results)


Filters:
- Query: Artificial Intelligence
- Author: Yann LeCun
- Title must contain: None
- Category: cs.LG
- Exclude words: Reinforcement Learning
- Abstract must contain: Machine Learning
- Journal Reference: None
- DOI: None
- Sort By: relevance
- Full Query: Artificial Intelligence AND au:"Yann LeCun" AND cat:"cs.LG" AND abs:"Machine Learning" AND NOT "Reinforcement Learning"

----------------------------------------------------------------------------------------------------

Title: Reverse Engineering Self-Supervised Learning
- Author(s): Ido Ben-Shaul, Ravid Shwartz-Ziv, Tomer Galanti, Shai Dekel, Yann LeCun
- Category: cs.LG
- Abstract (first 200 chars): Self-supervised learning (SSL) is a powerful tool in machine learning, but
understanding the learned representations and their underlying mechanisms
remains a challenge. This paper presents an in-dept...

Filter Matches:
- Author Match: ✅
- Title Match: ✅
- Category Match: ✅
- Abstract Match: ✅
- Exclusion (NOT): ✅
-----------